In [ ]:
import pandas as pd
df = pd.read_csv("2021jan_2024sept_cleaned.csv")

In [ ]:
# Tranche-level relevant fields (cleaned names)
tranche_fields = [
    'LPC_Tranche_ID',
    'Tranche_Amount',
    'Tranche_Active_Date',
    'Tranche_Maturity_Date',
    'Tranche_Amended',
    'Tranche_O_A',
    'Tranche_Currency',
    
    'LPC_Deal_ID',
    'Deal_Amount',
    'Deal_Active_Date',
    'Deal_Input_Date',
    'Deal_Amended',
    'Phase',
    'Deal_Purpose',

    'Borrower_Id',
    'Borrower_Name',
    'Major_Industry_Group',

    'Lender_Parent_Id',
    'Lender_Name',
    'Lender_Parent_Name',
    'Lender_Parent_Operating_Country'
]

# Filter the DataFrame
df_tranche = df[tranche_fields]

In [ ]:
print(df_tranche['Lender_Parent_Id'].unique()[:5])

In [ ]:
# Get the some unique Lender_Parent_Id values
some_ids = df_tranche['Lender_Parent_Id'].unique()[:5]

# Filter the DataFrame to include only those rows
df_sample = df_tranche[df_tranche['Lender_Parent_Id'].isin(some_ids)]

# Display the result
df_sample

In [ ]:
# one_sample = df_sample[df_sample['Lender_Parent_Id']== 105527.0]
# one_sample

In [ ]:
df_sample.to_csv("tranche_explore/df_sample.csv", index=False)

# structure panel quarter data

### Add quarter

In [ ]:
# Ensure date columns are in datetime format
df_sample["Tranche_Active_Date"] = pd.to_datetime(df_sample["Tranche_Active_Date"], errors="coerce")
df_sample["Tranche_Maturity_Date"] = pd.to_datetime(df_sample["Tranche_Maturity_Date"], errors="coerce")

# Rename for consistency with previous code
df_tranche = df_sample.copy()

# 📆 Step 1: Define quarterly periods (from Jan 2021 to Sep 2024)
quarterly_periods = pd.date_range(start="2021-01-01", end="2024-09-30", freq="Q")

In [ ]:
# 🧺 Step 2: Create a list to collect quarterly panel records
panel_records = []

# 🔁 Step 3: Loop through each quarterly period
for quarter_end in quarterly_periods:
    # Calculate quarter start
    quarter_start = pd.Timestamp(quarter_end) - pd.DateOffset(months=2) - pd.DateOffset(days=quarter_end.day - 1)
    
    # 🧹 Step 4: Filter rows where tranche is active during this quarter
    active_tranches = df_tranche[
        (df_tranche["Tranche_Active_Date"] <= quarter_end) &
        (df_tranche["Tranche_Maturity_Date"] >= quarter_start)
    ].copy()

    # 🏷️ Step 5: Add quarter labels
    active_tranches["quarter_start"] = quarter_start
    active_tranches["quarter_end"] = quarter_end

    # Collect into list
    panel_records.append(active_tranches)

# 📊 Step 6: Combine all quarters into a single panel dataframe
df_panel = pd.concat(panel_records).reset_index(drop=True)

# ✅ Step 7: Preview results
print(f"Total records in quarterly panel: {len(df_panel)}")
print("Sample records:")
display(df_panel.head())


### aggregate

In [ ]:
# Ensure datetime columns are properly formatted
df_panel["Tranche_Active_Date"] = pd.to_datetime(df_panel["Tranche_Active_Date"])
df_panel["Tranche_Maturity_Date"] = pd.to_datetime(df_panel["Tranche_Maturity_Date"])
df_panel["quarter_start"] = pd.to_datetime(df_panel["quarter_start"])
df_panel["quarter_end"] = pd.to_datetime(df_panel["quarter_end"])

# Step 1: Add flags for new and matured tranches
df_panel["is_new_tranche"] = (df_panel["Tranche_Active_Date"] >= df_panel["quarter_start"]) & (df_panel["Tranche_Active_Date"] <= df_panel["quarter_end"])
df_panel["is_matured_tranche"] = (df_panel["Tranche_Maturity_Date"] >= df_panel["quarter_start"]) & (df_panel["Tranche_Maturity_Date"] <= df_panel["quarter_end"])

# Step 2: Group and aggregate by quarter
quarterly_bank_summary = df_panel.groupby([
    "quarter_end",
    "Lender_Parent_Id",
    "Lender_Name",
    "Lender_Parent_Name",
    "Lender_Parent_Operating_Country"
]).agg(
    total_active_tranches=("LPC_Tranche_ID", "nunique"),
    total_new_tranches=("is_new_tranche", "sum"),
    total_matured_tranches=("is_matured_tranche", "sum"),
    total_amount_outstanding=("Tranche_Amount", "sum")
).reset_index()

# Preview result
quarterly_bank_summary

In [ ]:
quarterly_bank_summary.to_csv("tranche_explore/sample_quarterly_bank_summary.csv", index=False)